# 06 · Pre-filtering vs Post-filtering

> **Notebook type: THEORY ONLY**  
> There is no dedicated Colab practical for this concept in the source labs.  
> Use this notebook to learn the idea, then jump to related lab notebooks if linked below.

**Handbook:** [Pre-filtering vs Post-filtering](../retriever-analogy-handbook.html#pre-post)  
**Section:** Concept 06 · Narrowing the search


## Analogy

**Visa Checked at Check-in vs at Immigration**

> Same rule, applied at a different moment — and the second version is expensive and leaky.


## Concept

Pre-filtering is the airline checking your visa at the check-in desk. No visa, you never board, and the flight carries only people who are allowed to arrive. Post-filtering lets everyone fly, then immigration at the destination turns four out of five passengers straight around.

The numbers make the difference obvious. With ten thousand documents and a filter for HR and 2026, pre-filtering leaves one hundred and fifty documents and runs the similarity search only on those. Post-filtering runs the search across all ten thousand, returns the top five, and then discards four of them — leaving one result where you asked for five, while more valid HR documents that never made the top five sit unretrieved in the database.


## Mapping

| At the airport | In retrieval | Consequence |
| --- | --- | --- |
| Visa checked at check-in | Pre-filtering | Smaller, safer search space |
| Visa checked at arrival | Post-filtering | Wasted retrieval, weaker guarantees |
| Passengers who never boarded | Documents excluded before search | Never enter the candidate set at all |
| Turned around at the border | Results discarded after search | Already loaded, already ranked, then dropped |
| A near-empty arrivals hall | Too few final results | The classic post-filter failure |


## Reference snippet (not a full lab)

This is the handbook mini-example for orientation only.


In [ ]:
# Pre-filter: the vector DB applies the constraint during the search
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4, "filter": {"department": "HR", "year": 2026}},
)
documents = retriever.invoke("Show the HR leave policy for 2026.")

# Post-filter: search first, discard afterwards
documents = vector_store.similarity_search("Show the HR leave policy for 2026.", k=5)
allowed = [
    d for d in documents
    if d.metadata.get("department") == "HR" and d.metadata.get("year") == 2026
]


## Where the analogy breaks

Where the analogy breaks — At an airport, the passenger without a visa did genuinely travel — and that is exactly the security point people miss. Under post-filtering, restricted documents were really loaded into your candidate set and really scored. If any layer logs, caches or traces that set, the restricted content has left the boundary even though the user never saw it. Pre-filtering is not merely the faster option for access control; it is the only correct one.


## Related lab notebooks

- Filtering labs: [`05_metadata_filtering.ipynb`](05_metadata_filtering.ipynb)


## Self-check

1. Can you state the concept in one sentence without jargon?  
2. When would using this idea *hurt* retrieval quality?  
3. Which failure mode in the chooser table maps here (if any)?
